In [ ]:
from google.colab import drive
import os

def _drive_ready(mountpoint='/content/drive'):
    """진짜로 Drive 가 붙어 있는가.

    isdir('/content/drive/MyDrive') 만 보면 안 된다 — 마운트가 끊긴 자리에 같은 이름의
    폴더가 남으면 붙지도 않았는데 붙은 것처럼 보이고, 그 뒤 읽기·쓰기가 전부 조용히
    로컬 디스크로 새 나간다(한참 뒤 엉뚱한 데서 FileNotFoundError 로 터진다)."""
    return os.path.ismount(mountpoint) and any(
        os.path.isdir(os.path.join(mountpoint, n)) for n in ('MyDrive', 'My Drive')
    )


def _ensure_drive(mountpoint='/content/drive'):
    """이미 붙어 있으면 넘어가고, 아니면 붙인다. 마운트 자리에 찌꺼기가 남아
    mount 가 거부하는 경우("Mountpoint must not already contain files")까지 복구한다."""
    if _drive_ready(mountpoint):
        return
    try:
        drive.mount(mountpoint)
        return
    except ValueError:
        pass

    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    if not os.path.ismount(mountpoint):
        # 마운트가 아니니 여기 있는 건 Drive 원본이 아니라 로컬 찌꺼기다. 지우지 말고
        # 옆으로 치워 둔다 — 이 상태에서 저장된 파일이 있었다면 살릴 수 있게.
        # (rm -rf 로 지우는 방법은 쓰지 않는다. 마운트가 살아 있었다면 원본이 날아간다.)
        stale, n = mountpoint + '_stale', 1
        while os.path.exists(stale):
            n += 1
            stale = f'{mountpoint}_stale{n}'
        os.rename(mountpoint, stale)
        print(f'마운트가 아닌 찌꺼기 폴더를 {stale} 로 치웠습니다.')
    drive.mount(mountpoint, force_remount=True)


_ensure_drive()


# league(경기 종류) 열 추가 — 이제 보정용입니다

**2026-08-10부터 크롤링 노트북(`1. KBSA_크롤링_(당해년도_버전)_.ipynb`)이 크롤 시점에 `league`를 채웁니다.**
`record_detail`을 받은 직후 같은 루프 안에서 `box_score`를 한 번 더 받아 라운드와 구장을 같이 가져오므로,
새로 크롤한 경기는 여기를 돌릴 필요가 없습니다.

아래 셀은 이제 두 경우에만 씁니다:
1. 크롤 당시 `box_score` 요청이 실패해 `league`가 빈 경기 메꾸기 (크롤 로그에 ⚠ 로 찍힙니다)
2. 이 변경 이전에 크롤한 과거 시즌 보정

> 같은 기능이 크롤링 노트북의 `backfill_league_column` 에도 있습니다. 한쪽 파싱을 고치면
> 다른 쪽도 같이 고쳐야 두 벌이 어긋나지 않습니다. 크롤링 노트북 쪽은 `box_score` 의 dd 를
> 슬래시로 갈라 (라운드, 구장) 을 함께 돌려주는 `fetch_game_name_dd` 로 바뀌었습니다.


In [ ]:
import os
import re
import time
import random
import unicodedata
import requests
import pandas as pd
from bs4 import BeautifulSoup


# 구장 백필 셀(아래)과 같은 뿌리를 본다. 시즌 폴더는 이 아래 네 자리 숫자 폴더다.
DRIVE_ROOT = "/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)"
PBP_PATTERN = re.compile(r'^\d{4}_\d{2}_[a-z]+\.xlsx$')


def available_years(drive_root=DRIVE_ROOT):
    """drive_root 아래에 실제로 있는 시즌 폴더(네 자리 숫자)를 오름차순으로 돌려준다."""
    if not os.path.isdir(drive_root):
        raise FileNotFoundError(f"Drive 경로가 없습니다: {drive_root}  (마운트 셀부터 실행하세요)")
    return sorted(int(n) for n in os.listdir(drive_root)
                  if re.fullmatch(r'\d{4}', n) and os.path.isdir(os.path.join(drive_root, n)))


def fetch_league(game_idx, max_retries=3, retry_wait=30):
    """box_score 의 dl.game_name > dd 에서 라운드(리그전/16강전/결승전)를 뽑는다. 없으면 None.

    dd 는 '2026.06.06 09:30 / 신월 야구공원 / 리그전' 처럼 슬래시로 세 칸이다.
    칸 수가 3이 아니면 형식이 바뀐 것이므로 경고만 찍고 아무것도 돌려주지 않는다.

    ※ 크롤링 노트북의 fetch_game_name_dd 와 같은 파싱이다(그쪽은 구장까지 함께 돌려준다).
      두 벌이 어긋나면 시즌마다 라운드 표기가 달라지므로, 한쪽을 고치면 다른 쪽도 고칠 것.
    """
    url = f"https://www.korea-baseball.com/game/box_score?game_idx={game_idx}"

    res = None
    for attempt in range(max_retries):
        try:
            res = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
            break
        except requests.exceptions.ConnectionError as e:
            if attempt < max_retries - 1:
                print(f"  [{game_idx}] 연결 실패 ({attempt+1}/{max_retries}), {retry_wait}초 후 재시도...")
                time.sleep(retry_wait)
            else:
                print(f"  [{game_idx}] 연결 실패, 재시도 초과: {e}")
                return None
    if res is None:
        return None

    res.encoding = "utf-8"
    game_name_elem = BeautifulSoup(res.text, "html.parser").find("dl", class_="game_name")
    dd = game_name_elem.find("dd") if game_name_elem else None
    if not dd:
        return None

    text = dd.get_text(strip=True)
    parts = [p.strip() for p in text.split("/")]
    if len(parts) != 3:
        print(f"  ⚠ [{game_idx}] box_score dd 형식이 예상과 다릅니다(칸 {len(parts)}개): {text!r}")
        return None
    return parts[2] or None


def backfill_league_column(years=None, target_game_ids=None, drive_root=DRIVE_ROOT):
    """
    years:
      - 2024                한 시즌 (구장 백필 셀의 backfill_stadium_column(2024) 와 같은 사용법)
      - [2024, 2025]        여러 시즌
      - range(2020, 2027)   범위도 그대로 들어간다
      - None(기본값)        drive_root 아래에 폴더가 있는 시즌 전부

      폴더가 없는 연도는 경고를 찍고 건너뛴다. os.walk 는 없는 경로에서도 예외 없이
      조용히 끝나므로, 연도를 잘못 적으면 아무것도 안 하고 '완료'처럼 보인다.

    target_game_ids:
      - None(기본값): 대상 시즌의 전체 game_id
      - 단일 game_id (str 또는 int): 그 경기 하나만
      - game_id 리스트: 나열된 경기들만
      나머지 game_id의 기존 league/details 값은 건드리지 않는다.

    이미 league 값이 채워져 있는 game_id는 재요청 없이 건너뛴다.
    (중간에 멈췄다가 다시 실행해도 이어서 처리됨)
    """
    if years is None:
        years = available_years(drive_root)
        print(f"대상 시즌(폴더가 있는 해): {', '.join(map(str, years)) or '없음'}")
    elif isinstance(years, (str, int)):
        years = [years]

    for year in years:
        base_dir = os.path.join(drive_root, str(year))
        if not os.path.isdir(base_dir):
            print(f"[건너뜀] {year} - 시즌 폴더가 없습니다: {base_dir}")
            continue
        _backfill_league_year(int(year), base_dir, target_game_ids)


def _backfill_league_year(year, base_dir, target_game_ids):
    """한 시즌 처리. 연도를 늘리는 일은 backfill_league_column 쪽에서만 한다."""
    missing_log_file = os.path.join(base_dir, "league_missing_game_ids.txt")

    if target_game_ids is not None:
        if isinstance(target_game_ids, (str, int)):
            target_game_ids = [target_game_ids]
        target_game_ids = set(str(g) for g in target_game_ids)

    league_cache = {}
    missing_ids = []
    elapsed_log = {}

    print(f"\n{'='*60}\n[{year}] {base_dir}")

    for root, dirs, files in os.walk(base_dir):
        for fname in sorted(files):
            fname_nfc = unicodedata.normalize('NFC', fname)
            if not PBP_PATTERN.match(fname_nfc):
                continue

            fpath = os.path.join(root, fname)
            df = pd.read_excel(fpath, engine="openpyxl", dtype={"game_id": str})

            file_game_ids = df['game_id'].dropna().unique().tolist()
            if target_game_ids is not None:
                file_game_ids = [gid for gid in file_game_ids if gid in target_game_ids]
                if not file_game_ids:
                    continue

            if 'league' in df.columns:
                existing_league = (
                    df.dropna(subset=['game_id'])
                    .drop_duplicates(subset=['game_id'])
                    .set_index('game_id')['league']
                    .to_dict()
                )
            else:
                existing_league = {}

            for gid in file_game_ids:
                if gid in league_cache:
                    continue

                prior = existing_league.get(gid)
                if isinstance(prior, str) and prior.strip():
                    league_cache[gid] = prior
                    print(f"  [{gid}] league = {prior} (이미 완료됨, 스킵)")
                    continue

                start = time.time()
                league = fetch_league(gid)
                elapsed = time.time() - start
                elapsed_log[gid] = elapsed

                league_cache[gid] = league if league else ''
                if league:
                    print(f"  [{gid}] league = {league} ({elapsed:.2f}초)")
                else:
                    missing_ids.append(gid)
                    print(f"  [{gid}] league 정보를 찾지 못함 ({elapsed:.2f}초)")
                time.sleep(random.uniform(4, 5))

            if 'league' not in df.columns:
                df['league'] = ''

            mask = df['game_id'].isin(file_game_ids)
            df.loc[mask, 'league'] = df.loc[mask, 'game_id'].map(league_cache)
            df.loc[mask, 'details'] = (
                df.loc[mask, 'date'] + ' ' + df.loc[mask, 'time'] + ' ' +
                df.loc[mask, 'stadium'] + ' ' + df.loc[mask, 'league']
            )

            cols = list(df.columns)
            if cols.index('league') != cols.index('stadium') + 1:
                cols.remove('league')
                cols.insert(cols.index('stadium') + 1, 'league')
                df = df[cols]

            df.to_excel(fpath, index=False, engine="openpyxl")
            print(f"[완료] {fname_nfc}")

    if missing_ids:
        with open(missing_log_file, "w", encoding="utf-8") as f:
            f.write("\n".join(missing_ids))
        print(f"\n[{year}] league 정보를 찾지 못한 game_id {len(missing_ids)}개 -> {missing_log_file}")

    if elapsed_log:
        avg = sum(elapsed_log.values()) / len(elapsed_log)
        print(f"[{year}] 신규 요청 {len(elapsed_log)}건, 평균 {avg:.2f}초 / "
              f"최대 {max(elapsed_log.values()):.2f}초 / 최소 {min(elapsed_log.values()):.2f}초")
    else:
        print(f"[{year}] 새로 요청한 경기 없음(전부 이미 채워져 있음)")


# ==================== 실행 ====================
# backfill_league_column(2024)                                  # 한 시즌 전체
backfill_league_column([2020, 2021, 2022, 2023])                          # 여러 시즌
# backfill_league_column()                                      # 폴더가 있는 시즌 전부
# backfill_league_column(2026, target_game_ids=38143)           # 그 시즌의 단일 game_id
# backfill_league_column(2026, target_game_ids=[38818, 38819])  # 복수 game_id

# stadium(구장) 열 다시 받기 — 2020·2021·2023 (+ 2024·2025)

이 시즌들은 구장명이 **첫 공백에서 잘린 채** 저장돼 있다. `곤지암 팀업캠퍼스 제2구장`이 `곤지암`으로, `기장 현대차 드림 볼파크`가 `기장`으로 들어가 있는 식이다(2026-08-06 표본 대조로 확인). 2022·2026은 멀쩡하다.

피해가 두 가지다:

1. **같은 구장이 시즌별로 두 개로 갈린다** — `기장`(2020·21·23) vs `기장현대차드림볼파크`(2022·26)가 다른 구장으로 집계된다.
2. **다른 하위 구장이 뭉친다** — 곤지암 제1/제2, 횡성 제1/제2, 경주 1/2구장이 한 덩어리가 되고, `포항 생활체육 야구장`이 `포항`에 섞인다.

박스스코어의 `dl.game_name > dd` 는 `2020.07.05 09:30 / 곤지암 팀업캠퍼스 제2구장 / 리그전` 형태라, **`/` 로 갈랐을 때 가운데 토큰**이 온전한 구장명이다. league 를 받을 때 이미 읽는 그 `dd` 이므로 같은 요청으로 해결된다.

**2026-08-23 추가: 2024·2025 도 같은 상태였다.** full_tracked_2024·2025 로 파크팩터를 내 보니 그 두 시즌만 `곤지암`·`선샤인`·`기장`·`울산`·`원주`·`횡성`·`천안`·`제주`·`춘천`·`SSG` 같은 짧은 표기로 들어가 있었다. 그 결과 두 시즌에서만 나타나는 유령 구장 키가 17개 생기고 **1,932경기 중 583경기(30%)** 가 엉뚱한 구장으로 집계됐다(구장 55→72개). 짧은 표기로는 하위 구장을 되살릴 수 없어서(곤지암 제1/제2, 선샤인밀양 주/보조, 횡성 제1/제2, 울산중구/문수, 제주오라/동복) 별칭으로 때우지 않고 여기서 다시 받는다. 다 받은 뒤 노트북 2→3 을 재실행해 full_tracked 를 다시 굽고, `python scripts/export_park_factors.py` 를 돌리면 7시즌이 자동으로 들어온다.

아래 셀은 `stadium` 을 원본 값으로 덮어쓰고 `details` 를 다시 만든다. 체크포인트 파일에 진행 상황을 남기므로 중간에 끊겨도 이어서 돌릴 수 있고, `dry_run=True` 로 먼저 무엇이 바뀔지만 확인할 수 있다.

In [ ]:
# 이 셀만 따로 돌려도 되도록 필요한 것을 여기서 다 갖춘다(위 league 셀과 독립).
import json
import os
import re
import time
import random
import unicodedata
import requests
import pandas as pd
from bs4 import BeautifulSoup

PBP_PATTERN = re.compile(r'^\d{4}_\d{2}_[a-z]+\.xlsx$')
DRIVE_ROOT = "/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)"
STADIUM_FIX_YEARS = [2020, 2021, 2023, 2024, 2025]   # 구장명이 첫 공백에서 잘린 시즌
                                                     # (참고용 메모 — 실행은 아래에서 연도를 직접 준다)


def fetch_stadium(game_idx, max_retries=3, retry_wait=30):
    """box_score 의 dl.game_name > dd 에서 구장명을 뽑는다. 없으면 None.

    dd 예: '2020.07.05 09:30 / 곤지암 팀업캠퍼스 제2구장 / 리그전'
    '/' 로 갈랐을 때 **두 번째** 토큰이 구장이다. 종료 경기 중에는 뒤에
    '/ 경기시간 : 09:27 ~ 12:40 (소요시간 : 03:13)' 가 더 붙는 것이 있어서
    '가운데'로 잡으면 안 되고 인덱스 1로 고정해야 한다.
    """
    url = f"https://www.korea-baseball.com/game/box_score?game_idx={game_idx}"

    res = None
    for attempt in range(max_retries):
        try:
            res = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
            break
        except requests.exceptions.ConnectionError as e:
            if attempt < max_retries - 1:
                print(f"  [{game_idx}] 연결 실패 ({attempt+1}/{max_retries}), {retry_wait}초 후 재시도...")
                time.sleep(retry_wait)
            else:
                print(f"  [{game_idx}] 연결 실패, 재시도 초과: {e}")
                return None

    res.encoding = "utf-8"
    soup = BeautifulSoup(res.text, "html.parser")

    game_name_elem = soup.find("dl", class_="game_name")
    if not game_name_elem:
        return None
    dd = game_name_elem.find("dd")
    if not dd:
        return None

    parts = [p.strip() for p in dd.get_text(strip=True).split("/")]
    return parts[1] if len(parts) > 1 and parts[1] else None


def backfill_stadium_column(year, target_game_ids=None, dry_run=False,
                            sleep_range=(4, 5), checkpoint_every=20):
    """{year} 하위 PBP 파일의 stadium 열을 박스스코어 원본 값으로 다시 채운다.

    target_game_ids:
      - None(기본값): 그 시즌 전체
      - 단일 game_id 또는 리스트: 그 경기들만

    dry_run=True 면 요청은 하되 **파일은 건드리지 않고** 무엇이 바뀔지만 출력한다.
    먼저 몇 경기로 확인해 보는 용도다.

    이어서 돌리기: 받은 값을 {year}/stadium_backfill_{year}.json 에 계속 적어두고,
    다시 실행하면 거기 있는 game_id 는 재요청하지 않는다. league 백필과 달리
    '기존 값이 채워져 있으면 스킵'을 쓸 수 없다 — 지금 들어있는 값 자체가 잘린
    값이라, 채워져 있다는 사실만으로는 옳은지 알 수 없기 때문이다.

    sleep_range: 요청 간격. 기본은 league 백필과 같은 (9, 11)초라 한 시즌
    800경기면 두 시간 남짓 걸린다. 급하면 (2, 3) 으로 줄여도 되지만 그만큼
    상대 서버를 세게 두드린다.
    """
    base_dir = f"{DRIVE_ROOT}/{year}"
    ckpt_path = os.path.join(base_dir, f"stadium_backfill_{year}.json")

    if target_game_ids is not None:
        if isinstance(target_game_ids, (str, int)):
            target_game_ids = [target_game_ids]
        target_game_ids = set(str(g) for g in target_game_ids)

    stadium_cache = {}
    if os.path.exists(ckpt_path):
        with open(ckpt_path, encoding="utf-8") as f:
            stadium_cache = json.load(f)
        print(f"체크포인트에서 {len(stadium_cache)}건 이어받음: {ckpt_path}")

    def save_ckpt():
        with open(ckpt_path, "w", encoding="utf-8") as f:
            json.dump(stadium_cache, f, ensure_ascii=False, indent=1)

    missing_ids = []
    changes = {}          # (이전값, 새값) -> 경기 수
    n_fetched = 0
    n_files = 0

    for root, dirs, files in os.walk(base_dir):
        for fname in sorted(files):
            fname_nfc = unicodedata.normalize('NFC', fname)
            if not PBP_PATTERN.match(fname_nfc):
                continue

            fpath = os.path.join(root, fname)
            df = pd.read_excel(fpath, engine="openpyxl", dtype={"game_id": str})
            if 'stadium' not in df.columns:
                print(f"[건너뜀] {fname_nfc} — stadium 열 없음")
                continue

            file_game_ids = df['game_id'].dropna().unique().tolist()
            if target_game_ids is not None:
                file_game_ids = [gid for gid in file_game_ids if gid in target_game_ids]
                if not file_game_ids:
                    continue

            prev_stadium = (df.dropna(subset=['game_id'])
                              .drop_duplicates(subset=['game_id'])
                              .set_index('game_id')['stadium'].to_dict())

            for gid in file_game_ids:
                if gid in stadium_cache:
                    continue
                stadium = fetch_stadium(gid)
                n_fetched += 1
                if stadium:
                    stadium_cache[gid] = stadium
                    before = str(prev_stadium.get(gid, ''))
                    mark = "" if before == stadium else "  <-- 변경"
                    print(f"  [{gid}] {before} -> {stadium}{mark}")
                else:
                    missing_ids.append(gid)
                    print(f"  [{gid}] 구장 정보를 찾지 못함 — 기존 값 유지")
                if n_fetched % checkpoint_every == 0:
                    save_ckpt()
                time.sleep(random.uniform(*sleep_range))

            # 못 받은 경기는 기존 값을 그대로 둔다(빈 값으로 덮어쓰지 않는다).
            applied = {gid: stadium_cache[gid] for gid in file_game_ids if gid in stadium_cache}
            if not applied:
                continue
            for gid, new in applied.items():
                before = str(prev_stadium.get(gid, ''))
                if before != new:
                    changes[(before, new)] = changes.get((before, new), 0) + 1

            if dry_run:
                print(f"[dry-run] {fname_nfc} — {len(applied)}경기 확인만 하고 저장 안 함")
                continue

            mask = df['game_id'].isin(applied.keys())
            df.loc[mask, 'stadium'] = df.loc[mask, 'game_id'].map(applied)
            if {'date', 'time', 'league'}.issubset(df.columns):
                df.loc[mask, 'details'] = (
                    df.loc[mask, 'date'] + ' ' + df.loc[mask, 'time'] + ' ' +
                    df.loc[mask, 'stadium'] + ' ' + df.loc[mask, 'league']
                )
            df.to_excel(fpath, index=False, engine="openpyxl")
            n_files += 1
            print(f"[완료] {fname_nfc}")

    save_ckpt()

    print(f"\n{'='*60}\n{year} 요약 — 신규 요청 {n_fetched}건 / 저장한 파일 {n_files}개")
    if changes:
        print(f"바뀐 구장명 {len(changes)}종 (경기 수 기준 내림차순)")
        for (before, new), cnt in sorted(changes.items(), key=lambda kv: -kv[1]):
            print(f"  {cnt:4d}경기  '{before}' -> '{new}'")
    else:
        print("바뀐 구장명 없음")
    if missing_ids:
        print(f"구장 정보를 못 받은 game_id {len(missing_ids)}개: {missing_ids[:20]}")


# ==================== 실행 ====================
# 1) 먼저 몇 경기로 확인 (파일 안 건드림)
# backfill_stadium_column(2020, target_game_ids=[22023, 22024], dry_run=True)

# 2) 한 시즌씩 진행
backfill_stadium_column(2024)
backfill_stadium_column(2025)